# Stochastic Calculus, Step by Step · Lesson 26: Risk-neutral pricing

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Where a lesson needs the share's real-world expected return, it is μ = 8%. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. The discounted share becomes a martingale · 3. Every self-financing portfolio too · 4. The pricing formula · 5. Where the hedge comes from · 6. The two fundamental theorems · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 26,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
mu = 0.08                         # the share's real-world expected return
theta_mpr = (mu - r) / sigma      # the market price of risk
Nv = np.vectorize(N)              # N, for arrays
phi = lambda x: np.exp(-np.asarray(x, float) ** 2 / 2) / math.sqrt(2 * math.pi)   # the normal density


def d_plus(s, tau, sig=sigma, k=K):
    return (np.log(s / k) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))


def d_minus(s, tau, sig=sigma, k=K):
    return d_plus(s, tau, sig, k) - sig * np.sqrt(tau)


def call(s, tau, sig=sigma, k=K):
    """The Black-Scholes call price: Lesson 20 derives it."""
    return s * Nv(d_plus(s, tau, sig, k)) - k * np.exp(-r * tau) * Nv(d_minus(s, tau, sig, k))


def put(s, tau, sig=sigma, k=K):
    return call(s, tau, sig, k) - s + k * np.exp(-r * tau)


out["bs"] = {"mu": mu, "theta": theta_mpr, "d_plus": float(d_plus(S0, T)), "d_minus": float(d_minus(S0, T)),
             "N_plus": N(float(d_plus(S0, T))), "N_minus": N(float(d_minus(S0, T))),
             "call": float(call(S0, T)), "put": float(put(S0, T))}
out["bs"]

## 2. The discounted share becomes a martingale

With $D_t = e^{-rt}$, the product rule gives $d(D_tS_t) = \sigma D_tS_t(\theta\,dt + dW_t)$, with $\theta = (\mu - r)/\sigma$. Girsanov (Lesson 25) turns $\theta\,dt + dW$ into $d\tilde W$:

$$d(D_tS_t) = \sigma D_tS_t\,d\tilde W_t .$$

No $dt$ term, so under $\tilde P$ it is a martingale (Lesson 18). Simulated with $\mu = 8\%$, and weighted by $Z_T$:

In [ ]:
rng = np.random.default_rng(seed)
paths, n_t = 400_000, 12
dt = T / n_t
tg = dt * np.arange(1, n_t + 1)
dW = rng.normal(0.0, math.sqrt(dt), (paths, n_t))
W = dW.cumsum(axis=1)
S = S0 * np.exp((mu - 0.5 * sigma**2) * tg + sigma * W)
Z = np.exp(-theta_mpr * W[:, -1] - 0.5 * theta_mpr**2 * T)
disc = np.exp(-r * tg)
DS = disc * S
wE = lambda x: float((Z * x).mean())
mart_rows = [{"t": float(tg[i]), "P_mean": float(DS[:, i].mean()), "P_exact": S0 * math.exp((mu - r) * tg[i]),
              "Q_mean": wE(DS[:, i]), "Q_exact": S0} for i in (2, 5, 8, 11)]
out["mart"] = {"rows": mart_rows, "sharpe": theta_mpr, "excess": mu - r}
pd.DataFrame(mart_rows)

## 3. Every self-financing portfolio too

Lesson 19's wealth equation discounts to $d(D_tX_t) = \Delta_t\,d(D_tS_t) = \Delta_t\sigma D_tS_t\,d\tilde W_t$: again no $dt$ term, whatever $\Delta_t$ you choose, as long as it doesn't look into the future. So no trading strategy changes the $\tilde P$ average of discounted wealth. Four strategies, none of which beats the cash account under $\tilde P$:

In [ ]:
def wealth(rule):
    # Self-financing wealth on the simulated paths, start 10, monthly rebalance.
    X = np.full(paths, 10.0)
    s_prev = np.full(paths, S0)
    for i in range(n_t):
        pos = rule(s_prev, i * dt)
        s_new = S[:, i]
        X = pos * s_new + (X - pos * s_prev) * math.exp(r * dt)
        s_prev = s_new
    return X


rules = [("Cash only", lambda s, t_: 0 * s), ("Hold 1 share", lambda s, t_: 1 + 0 * s),
         ("Momentum: 2 shares above 100, none below", lambda s, t_: 2.0 * (s > S0)),
         ("Contrarian: 2 shares below 100, none above", lambda s, t_: 2.0 * (s < S0))]
strat_rows = []
for name, rule in rules:
    X_T = wealth(rule)
    strat_rows.append({"name": name, "real": float(X_T.mean() * math.exp(-r * T)), "rn": wE(X_T * math.exp(-r * T))})
out["strat"] = {"rows": strat_rows}
pd.DataFrame(strat_rows)

## 4. The pricing formula

If a portfolio replicates a payoff $V_T$, its discounted value is a martingale, so its value today is

$$V_t = \tilde E\big[e^{-r(T - t)}V_T \,\big|\, \mathcal F_t\big] .$$

Compared four ways, with one wrong answer to show why the probability matters:

In [ ]:
gen = np.random.default_rng(seed + 1)
Zg = gen.standard_normal(2_000_000)
disc_T = math.exp(-r * T)
pay_rn = disc_T * np.maximum(S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * Zg) - K, 0)
pay_real = disc_T * np.maximum(S0 * np.exp((mu - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * Zg) - K, 0)
theta_ = theta_mpr
W_g = math.sqrt(T) * Zg
pay_wt = pay_real * np.exp(-theta_ * W_g - 0.5 * theta_**2 * T)
price_rows = [{"way": "The Black-Scholes formula", "price": float(call(S0, T)), "se": 0.0},
              {"way": "Simulate under the risk-neutral probability", "price": float(pay_rn.mean()), "se": float(pay_rn.std() / math.sqrt(Zg.size))},
              {"way": "Simulate under the real probability, weight by Z", "price": float(pay_wt.mean()), "se": float(pay_wt.std() / math.sqrt(Zg.size))},
              {"way": "Simulate under the real probability, no weights (wrong)", "price": float(pay_real.mean()), "se": float(pay_real.std() / math.sqrt(Zg.size))}]
sg = np.linspace(40, 220, 361)
dens = lambda s, m_: phi((np.log(s / S0) - (m_ - 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))) / (s * sigma * math.sqrt(T))
out["price"] = {"rows": price_rows, "gap": float(pay_real.mean() - call(S0, T)),
                "fig": {"S": sg.tolist(), "real": dens(sg, mu).tolist(), "rn": dens(sg, r).tolist()},
                "mean_real": S0 * math.exp(mu * T), "mean_rn": S0 * math.exp(r * T)}
pd.DataFrame(price_rows)

## 5. Where the hedge comes from

$M_t = \tilde E[D_TV_T \mid \mathcal F_t] = D_tV_t$ is a $\tilde P$-martingale. The **martingale representation theorem** says that any such martingale, in a world driven by one Brownian motion, can be written $M_t = M_0 + \int_0^t\Gamma_u\,d\tilde W_u$. Matching $\Gamma_u\,d\tilde W_u$ with $\Delta_u\sigma D_uS_u\,d\tilde W_u$ gives the hedge: $\Delta_u = \Gamma_u/(\sigma D_uS_u)$.

For the call, $\Gamma_u = \sigma D_uS_uN(d_+)$, the delta of Lesson 21. Rebuilding the martingale from its integral along one path:

In [ ]:
gen = np.random.default_rng(seed + 1)     # a path that finishes in the money
n_f = 5000
dt_f = T / n_f
dWt = gen.normal(0.0, math.sqrt(dt_f), n_f)
tf = dt_f * np.arange(n_f + 1)
Wtp = np.concatenate([[0.0], dWt.cumsum()])
Sp = S0 * np.exp((r - 0.5 * sigma**2) * tf + sigma * Wtp)
tau_f = T - tf[:-1]
M = np.exp(-r * tf[:-1]) * call(Sp[:-1], tau_f)
G = sigma * np.exp(-r * tf[:-1]) * Sp[:-1] * Nv(d_plus(Sp[:-1], tau_f))
I_ = (G * dWt).cumsum()
idx = [999, 2499, 3999, n_f - 2]
rep_rows = [{"t": float(tf[i + 1]), "M": float(np.exp(-r * tf[i + 1]) * call(Sp[i + 1], T - tf[i + 1])), "rebuilt": float(M[0] + I_[i])} for i in idx]
final = math.exp(-r * T) * max(Sp[-1] - K, 0)
out["rep"] = {"rows": rep_rows, "M0": float(M[0]), "final_payoff": final, "final_rebuilt": float(M[0] + I_[-1]), "S_end": float(Sp[-1]),
              "gap": float(M[0] + I_[-1] - final)}
pd.DataFrame(rep_rows)

## 6. The two fundamental theorems

**First:** a risk-neutral probability exists means no arbitrage. **Second:** it is unique means every payoff can be hedged (the market is complete). In the model here, one share and one Brownian motion: exactly one $\theta$, so one $\tilde P$, so one price. Two risky assets on the same Brownian motion must have the same $\theta$, or there is an arbitrage: a portfolio of the two with no risk and more than $r$.

In [ ]:
# Two assets driven by the same W, with different market prices of risk.
mu1, s1, mu2, s2 = 0.08, 0.20, 0.14, 0.30
th1, th2 = (mu1 - r) / s1, (mu2 - r) / s2
w1 = s2 / (s2 - s1)   # weight in asset 1 per unit of wealth, the rest in asset 2, cancels the dW term
w2 = 1 - w1
drift_port = w1 * mu1 + w2 * mu2
vol_port = w1 * s1 + w2 * s2
out["ftap"] = {"mu1": mu1, "s1": s1, "th1": th1, "mu2": mu2, "s2": s2, "th2": th2, "w1": w1, "w2": w2, "drift": drift_port, "vol": vol_port, "excess": drift_port - r,
               "mu2_fair": r + th1 * s2}
out["ftap"]

In [ ]:
# Chart data for the video (no random numbers)
out["price"]["fig"]["ticks"] = [{"x": float(x_), "label": f"{x_:.0f}"} for x_ in (60, 100, 140, 180, 220)]

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 4: Solutions*.

**Exercise 1.** Use the product rule on $D_tS_t$ to derive $d(D_tS_t) = \sigma D_tS_t(\theta\,dt + dW_t)$, with $\theta = (\mu - r)/\sigma$. Why is there no cross-variation term between $D$ and $S$?

**Exercise 2.** $\theta$ is the share's **Sharpe ratio**, excess return per unit of risk. Evaluate it with the course's numbers. What does $\theta = 0$ mean, and what would $\tilde P$ be then? What is $E[S_T]$ under $\tilde P$ and under $P$?

**Exercise 3.** Show that $d(D_tX_t) = \Delta_t\,d(D_tS_t)$ for a self-financing portfolio, and therefore that $D_tX_t$ is a $\tilde P$-martingale. Why does this rule out making money for sure, whatever $\Delta_t$ is?

**Exercise 4.** Use the pricing formula to find the value today of a forward contract paying $S_T - F$ at $T$. Which $F$ makes it worth zero? Evaluate it with the course's numbers. Did you need the volatility? What about $\mu$?

**Exercise 5.** Find the martingale representation of $\tilde W_T^2 - T$: use Itô's lemma to write it as $\int_0^T2\tilde W_t\,d\tilde W_t$. If this were the discounted value of a claim, how many shares would the hedge hold at time $t$, with $r = 0$?

**Exercise 6.** Two assets are driven by one Brownian motion: $(\mu_1, \sigma_1) = (8\%, 20\%)$ and $(\mu_2, \sigma_2) = (14\%, 30\%)$. Build the portfolio that removes the $dW$ term. What does it earn? What arbitrage does that allow? Which $\mu_2$ removes it?

**Exercise 7.** (Code) Price the call and a cash digital on the same paths under $P$ with $\mu = 8\%$, weighting by $Z_T$, and under $\tilde P$ directly. Compare standard errors for each with the formula. Which suffers more from the weights, and why?

**Exercise 8.** (Code) Check the representation for the call on 200 paths of 2000 steps: compute $M_0 + \sum\Gamma\,\Delta\tilde W$ and compare with the discounted payoff path by path. Report the root-mean-square gap at 500, 1000 and 2000 steps. How fast does it shrink?

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")